# Run ALPR Vehicle System in Google Colab

Use this notebook to run the ALPR & Vehicle Classification dashboard on a Colab GPU runtime. 

**Recommended runtime:** Go to **Runtime → Change runtime type** and select **T4 GPU**.

In [ ]:
# 1) Verify GPU acceleration is active
!nvidia-smi

import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU Type:', torch.cuda.get_device_name(0))

## Option A: Clone directly from GitHub (Recommended)

Clone the project repository directly into Colab. This ensures you always run the latest committed version without needing to zip/upload.

In [ ]:
# 2A) Clone repo
%cd /content
!rm -rf /content/ALPR-Vehicle-System

# Clone the repository
!git clone https://github.com/x0uls/ALPR-Vehicle-System.git

PROJECT_DIR = '/content/ALPR-Vehicle-System'
%cd $PROJECT_DIR
!ls -la

## Option B: Upload Project ZIP Directly

Fallback option to upload a local `alpr-vehicle-system.zip` file directly to Colab.

In [ ]:
# 2B) Upload and Extract project ZIP
%cd /content
!rm -rf /content/alpr-vehicle-system*

from google.colab import files
uploaded = files.upload()

import os, zipfile
zip_names = [name for name in uploaded if name.lower().endswith('.zip')]
assert zip_names, 'Please upload a .zip file of the project.'

zip_path = zip_names[0]
extract_dir = '/content/alpr-vehicle-system'
os.makedirs(extract_dir, exist_ok=True)
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_dir)

# Auto-detect folder nesting inside the ZIP
candidates = []
for root, dirs, files_in_root in os.walk(extract_dir):
    if 'app.py' in files_in_root and 'src' in dirs:
        candidates.append(root)
assert candidates, 'Could not find app.py and src/ folder inside the zip.'

PROJECT_DIR = candidates[0]
%cd $PROJECT_DIR
!ls -la

## Environment Setup & File Check

In [ ]:
# 3) Install dependencies
!sudo add-apt-repository -y ppa:alex-p/tesseract-ocr5
!sudo apt-get update -qq
!sudo apt-get install -y tesseract-ocr tesseract-ocr-eng

# Tesseract 5 path localization fix for trained data files
!sudo mkdir -p /usr/share/tesseract-ocr/5/tessdata
!sudo find / -name "eng.traineddata" 2>/dev/null -exec cp {} /usr/share/tesseract-ocr/5/tessdata/ \;

!pip install -q ultralytics easyocr pandas openpyxl uvicorn fastapi pyngrok python-multipart jinja2 pytesseract

!tesseract --version
!tesseract --list-langs

In [ ]:
# 4) Verify required structure & cache models (pre-download)
from pathlib import Path
required_paths = [
    Path('app.py'),
    Path('src/pipeline.py'),
    Path('src/templates/index.html'),
    Path('models/yolo_plate/best.pt')
]
missing = [str(p) for p in required_paths if not p.exists()]
if missing:
    raise FileNotFoundError('Missing required project files:\n' + '\n'.join(missing))

print('Downloading base YOLOv8 model if not cached...')
if not Path('yolov8n.pt').exists():
    from ultralytics import YOLO
    YOLO('yolov8n.pt')

print('Downloading EasyOCR models (detection/recognition) if not cached...')
import easyocr
import torch
easyocr.Reader(['en'], gpu=torch.cuda.is_available())

print('✅ Project structure ready and all models cached.')

## Launch Interactive Web Dashboard

In [ ]:
# 5) Start Server & Open Public Tunnel
import threading, time, os, subprocess

PORT = 7860

# Clear previous logs
!rm -f uvicorn.log

# Run uvicorn in background thread and redirect output to uvicorn.log
def _run():
    os.system(f'python -m uvicorn app:app --host 0.0.0.0 --port {PORT} > uvicorn.log 2>&1')

threading.Thread(target=_run, daemon=True).start()
time.sleep(5) # Give the server a few seconds to initialize

# Download and launch Cloudflare tunnel for free instant sharing
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared

tunnel = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', f'http://localhost:{PORT}'],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True
)

# Extract and display tunnel URLs
import re
for line in tunnel.stderr:
    match = re.search(r'(https://[a-z0-9-]+\.trycloudflare\.com)', line)
    if match:
        url = match.group(1)
        print(f'\n✅ Web Dashboard: {url}')
        break

print('\n💡 Note: If you get a 502 Bad Gateway error when opening the link above,')
print('   run "!cat uvicorn.log" in a new cell to see what error occurred.')

# Keep cell alive to service tunnel connection
try:
    while True:
        time.sleep(1)
except KeyboardInterrupt:
    tunnel.kill()
    print('Tunnel shut down.')